In [6]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = ["glioma", "meningioma", "notumor", "pituitary"]

MODEL_PATH = "/content/brain_tumor_final_v2.pth"

# ============================================================
# RECONSTRUIRE EXACTEMENT LE MODELE TRAINING
# ============================================================
model = models.efficientnet_b0(weights=None)

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 256),   # 🔥 couche cachée (IMPORTANT)
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 4)       # sortie finale
)

model = model.to(device)

# ============================================================
# CHARGEMENT
# ============================================================
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("✅ Modèle chargé correctement")

# ============================================================
# TRANSFORMATIONS
# ============================================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ============================================================
# PREDICTION
# ============================================================
def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

    print("\n==============================")
    print("🔬 RESULTAT FINAL")
    print("==============================")
    print("Classe :", class_names[pred.item()])
    print("Confiance :", round(conf.item()*100, 2), "%")
    print("==============================")

    return class_names[pred.item()], conf.item()

# TEST
predict("/content/Tr-no_6.jpg")

✅ Modèle chargé correctement

🔬 RESULTAT FINAL
Classe : notumor
Confiance : 79.03 %


('notumor', 0.7903249859809875)